# Lesson 08 - Explicit graph workflows with edges

**Level:** advanced | **Demo time:** 15 minutes | **Builds on:** Lessons 03 and 06

Some support processes need an application-owned route rather than an agent-owned conversation. In this lesson, a Foundry-backed classifier produces validated data, ordinary Python selects one branch, and every branch converges on a shared finalizer.

This is the Microsoft Agent Framework **general graph API** (`agent-framework-core` 1.19.0), not an orchestration builder.

## Learning objectives

- Define typed workflow messages with dataclasses and Pydantic.
- Wrap a Foundry-backed agent in an `Executor` with `@handler`.
- Connect executors with `add_edge` and `add_switch_case_edge_group`.
- Stream executor lifecycle events and a typed workflow output.
- Explain when an explicit graph is preferable to `HandoffBuilder`.

## Explicit graph versus handoffs

| General graph API | `HandoffBuilder` |
|---|---|
| Application code owns every edge and branch condition | Agents choose when to transfer the conversation |
| Executors exchange typed application messages | Participants exchange conversational messages |
| Best for deterministic stages, policy gates, and auditable routing | Best when specialists need to speak directly with the customer |
| A finalizer can guarantee one stable output contract | The conversation itself is usually the primary result |

The topology for this lesson is deliberately visible:

```text
SupportRequest
      |
      v
classify_ticket  -- Foundry agent returns TicketClassification
      |
      v
Case / Case / Case / Default
   |       |       |       |
 human   order   returns  general
   |       |       |       |
   +-------+-------+-------+
                  |
                  v
             finalize_plan  --> SupportPlan
```

The first matching `Case` wins. The `Default` branch makes the route total, so an unexpected or newly added category still reaches a safe handler.

In [ ]:
from dataclasses import dataclass
from typing import Literal, Never

from agent_framework import (
    Agent,
    Case,
    Default,
    Executor,
    Workflow,
    WorkflowBuilder,
    WorkflowContext,
    handler,
)
from pydantic import BaseModel, Field

SupportCategory = Literal["order_status", "return_or_refund", "technical", "other"]
Priority = Literal["low", "normal", "high"]


@dataclass(frozen=True)
class SupportRequest:
    customer_id: str
    message: str


class TicketClassification(BaseModel):
    category: SupportCategory
    priority: Priority
    summary: str = Field(description="One-sentence summary grounded only in the customer message.")
    order_id: str | None = Field(default=None, description="Order ID exactly as supplied by the customer.")
    requires_human: bool = Field(
        description="True for safety, fraud, payment disputes, or identity/authorization decisions."
    )


@dataclass(frozen=True)
class ClassifiedTicket:
    request: SupportRequest
    classification: TicketClassification


@dataclass(frozen=True)
class BranchDecision:
    ticket: ClassifiedTicket
    queue: str
    owner: str
    next_action: str


class SupportPlan(BaseModel):
    customer_id: str
    category: SupportCategory
    priority: Priority
    queue: str
    owner: str
    next_action: str
    summary: str
    order_id: str | None = None

## Executors are typed graph nodes

`Executor` subclasses expose one or more `@handler` methods. MAF discovers the accepted message type from the handler parameter and the emitted types from `WorkflowContext`:

- `WorkflowContext[ClassifiedTicket]` permits `send_message(ClassifiedTicket(...))` to downstream edges.
- `WorkflowContext[Never, SupportPlan]` declares no downstream message and a `SupportPlan` workflow output.
- `yield_output` creates a caller-facing workflow event; `send_message` continues graph execution.

The classifier executor is the only node that calls the model. Branch executors remain deterministic, inexpensive, and easy to unit test.

In [ ]:
class TicketClassifierExecutor(Executor):
    def __init__(self, classifier_agent: Agent) -> None:
        super().__init__(id="classify_ticket")
        self._classifier_agent = classifier_agent

    @handler
    async def classify(
        self,
        request: SupportRequest,
        ctx: WorkflowContext[ClassifiedTicket],
    ) -> None:
        response = await self._classifier_agent.run(
            request.message,
            options={"response_format": TicketClassification},
        )
        classification = response.value
        if not isinstance(classification, TicketClassification):
            raise RuntimeError("The classifier returned no validated TicketClassification.")

        await ctx.send_message(
            ClassifiedTicket(request=request, classification=classification)
        )


class HumanEscalationBranch(Executor):
    @handler
    async def handle(
        self,
        ticket: ClassifiedTicket,
        ctx: WorkflowContext[BranchDecision],
    ) -> None:
        await ctx.send_message(
            BranchDecision(
                ticket=ticket,
                queue="human-review",
                owner="senior-support",
                next_action="Verify identity, review risk, and decide the authorized remedy.",
            )
        )


class OrderStatusBranch(Executor):
    @handler
    async def handle(
        self,
        ticket: ClassifiedTicket,
        ctx: WorkflowContext[BranchDecision],
    ) -> None:
        await ctx.send_message(
            BranchDecision(
                ticket=ticket,
                queue="fulfillment",
                owner="order-specialist",
                next_action="Look up shipment status and explain the latest verified scan.",
            )
        )


class ReturnsAndRefundsBranch(Executor):
    @handler
    async def handle(
        self,
        ticket: ClassifiedTicket,
        ctx: WorkflowContext[BranchDecision],
    ) -> None:
        await ctx.send_message(
            BranchDecision(
                ticket=ticket,
                queue="returns-and-refunds",
                owner="returns-specialist",
                next_action="Check policy and confirm intent before creating any request.",
            )
        )


class GeneralSupportBranch(Executor):
    @handler
    async def handle(
        self,
        ticket: ClassifiedTicket,
        ctx: WorkflowContext[BranchDecision],
    ) -> None:
        await ctx.send_message(
            BranchDecision(
                ticket=ticket,
                queue="general-support",
                owner="support-generalist",
                next_action="Collect missing details and provide product troubleshooting.",
            )
        )


class PlanFinalizer(Executor):
    @handler
    async def finalize(
        self,
        decision: BranchDecision,
        ctx: WorkflowContext[Never, SupportPlan],
    ) -> None:
        ticket = decision.ticket
        classification = ticket.classification
        await ctx.yield_output(
            SupportPlan(
                customer_id=ticket.request.customer_id,
                category=classification.category,
                priority=classification.priority,
                queue=decision.queue,
                owner=decision.owner,
                next_action=decision.next_action,
                summary=classification.summary,
                order_id=classification.order_id,
            )
        )

## Assemble the graph

`add_switch_case_edge_group` evaluates cases in order. Human review is intentionally first so a risky billing message cannot be routed around the escalation policy merely because it also mentions a refund.

Each branch uses `add_edge` to send its `BranchDecision` to the same finalizer. This is a merge point, not a barrier: only the selected branch runs, so the finalizer receives one message.

`max_iterations=8` is a safety ceiling on graph convergence. The initial executor run is the first iteration and later supersteps advance messages across edges. MAF graphs may contain cycles, which is useful for bounded review/retry loops, but every cycle still needs an explicit exit condition. `max_iterations` protects against a non-converging graph; it is not a substitute for business termination logic.

In [ ]:
def build_support_workflow(classifier_agent: Agent) -> Workflow:
    classifier = TicketClassifierExecutor(classifier_agent)
    human = HumanEscalationBranch(id="human_escalation")
    order = OrderStatusBranch(id="order_status")
    returns = ReturnsAndRefundsBranch(id="returns_refunds")
    general = GeneralSupportBranch(id="general_support")
    finalizer = PlanFinalizer(id="finalize_plan")

    builder = WorkflowBuilder(
        name="customer_support_graph",
        description="Classify a customer request, select one support branch, and emit a support plan.",
        start_executor=classifier,
        max_iterations=8,
        output_from=[finalizer],
    )

    builder.add_switch_case_edge_group(
        classifier,
        [
            Case(
                condition=lambda ticket: ticket.classification.requires_human,
                target=human,
            ),
            Case(
                condition=lambda ticket: ticket.classification.category == "order_status",
                target=order,
            ),
            Case(
                condition=lambda ticket: ticket.classification.category == "return_or_refund",
                target=returns,
            ),
            Default(target=general),
        ],
    )
    builder.add_edge(human, finalizer)
    builder.add_edge(order, finalizer)
    builder.add_edge(returns, finalizer)
    builder.add_edge(general, finalizer)

    return builder.build()

## Create the Foundry-backed classifier

The workshop helper keeps configuration and Microsoft Entra authentication consistent with earlier lessons. Constructing the graph does not call Azure; the model call occurs only when `classify_ticket` handles a `SupportRequest`.

The agent produces the Pydantic response format, while the executor converts that model-facing result into the graph's application message.

In [ ]:
from maf_foundry_hackathon.config import (
    WorkshopSettings,
    create_chat_client,
    create_credential,
)

settings = WorkshopSettings.from_env()
credential = create_credential()
chat_client = create_chat_client(settings, credential)

classifier_agent = Agent(
    client=chat_client,
    name="support_ticket_classifier",
    description="Classifies customer-support messages for an explicit workflow graph.",
    instructions=(
        "Classify only the supplied customer message. Use order_status for shipping or tracking; "
        "return_or_refund for damaged items, returns, refunds, or duplicate charges; technical for "
        "product troubleshooting; and other when none applies. Use high priority for safety, fraud, "
        "or a complete outage. Set requires_human for safety, suspected fraud, payment disputes, "
        "or identity and authorization decisions. Never invent an order ID."
    ),
)

support_workflow = build_support_workflow(classifier_agent)
print(f"Built {support_workflow.name!r} with max_iterations={support_workflow.max_iterations}.")

## Run and observe the graph

`workflow.run(..., stream=True)` returns a `ResponseStream`. Executor lifecycle events make the selected path visible, and the finalizer's `yield_output` produces an `output` event. After consuming the stream, `get_final_response()` returns a `WorkflowRunResult`; `get_outputs()` extracts the typed outputs without parsing event text.

Try changing the message to an order-tracking or technical question and compare the invoked branch.

In [ ]:
request = SupportRequest(
    customer_id="CUST-042",
    message="I was charged twice for order ORD-1001. Please refund the duplicate charge today.",
)

stream = support_workflow.run(request, stream=True)

async for event in stream:
    if event.type in {"executor_invoked", "executor_completed"}:
        print(f"{event.type:>20} | {event.executor_id}")
    elif event.type == "output":
        print(f"{'workflow_output':>20} | {event.executor_id}")
        if isinstance(event.data, BaseModel):
            print(event.data.model_dump_json(indent=2))
        else:
            print(event.data)

result = await stream.get_final_response()
outputs = result.get_outputs()
if not outputs:
    raise RuntimeError("The workflow completed without a SupportPlan output.")

support_plan = outputs[-1]
support_plan

## Presenter talking points and production notes

- The model recommends a classification; code owns the route.
- Put authorization and transaction rules inside trusted services, not in model instructions.
- Keep high-risk cases ahead of category cases because switch conditions are first-match-wins.
- Add tools inside the selected branch only when that branch needs them.
- If you add a retry edge back to classification, include a typed retry counter and a normal exit path in addition to `max_iterations`.
- Evaluate classification accuracy separately from branch completion and policy adherence.

Use `HandoffBuilder` when specialists should decide conversational ownership. Use this explicit graph when the application must explain exactly why a node ran and what typed data crossed each edge.

In [ ]:
credential.close()
print("Credential resources closed.")

## Recap

The workflow now has an explicit, typed topology: a Foundry-backed classifier executor, deterministic `Case`/`Default` branches, `add_edge` merges, and one final `SupportPlan`. The graph can evolve with additional nodes or bounded cycles without giving up application-owned control.